# Load and animate effective potentials, filling factors, and chemical potential from `sweeppone.ipynb`

This notebook is a lightweight post-processing companion for `sweeppone.ipynb`.

The original notebook performs a magnetic-field sweep, saving each self-consistent simulation in a separate subfolder of `sweep_fold/`. Each subfolder is expected to contain:

- `metadata.json`, with the simulation parameters, including the magnetic field `B`.
- `data.npz`, with numerical arrays, including the self-consistent effective potential `U` and, when available, the filling factor `nu`.

This notebook loads the effective potential, the filling factor, the chemical potential, and the corresponding magnetic field for each run, stores them in compact NumPy arrays, plots selected frames, exports a video of the potential sweep, exports a video of the filling-factor sweep, and plots the chemical potential during the B sweep.


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import HTML, display, Video

plt.rcParams.update({"font.size": 12})

## User options

Set `SWEEP_DIR` to the folder generated by `sweeppone.ipynb`. By default it is assumed to be next to this notebook.

`POTENTIAL_KEY = "U"` selects the self-consistent effective potential saved by the simulation. You can switch to `"U_ext"` if you want the bare donor/external potential instead.

`GROUND_EACH_FRAME = True` subtracts the minimum of each potential frame, matching the convention used in the original notebook when it computes `U_grounded = U - np.nanmin(U)`.


In [ ]:
SWEEP_DIR = Path("data_sweep")
POTENTIAL_KEY = "U"          # normally the self-consistent effective potential
FILLING_KEY = "nu"           # filling factor saved by the sweep notebook
GROUND_EACH_FRAME = True     # plot U - min(U) for each B
DTYPE = np.float32            # reduces memory usage

# Output files
CACHE_FILE = Path("output_sweep/effective_potential_and_filling_sweep.npz")
VIDEO_MP4 = Path("output_sweep/effective_potential_sweep.mp4")
VIDEO_GIF = Path("output_sweep/effective_potential_sweep.gif")
FILLING_VIDEO_MP4 = Path("output_sweep/filling_factor_sweep.mp4")
FILLING_VIDEO_GIF = Path("output_sweep/filling_factor_sweep.gif")
MU_PLOT_PNG = Path("output_sweep/chemical_potential_vs_B.png")

# Animation options
FPS = 10
DPI = 150
CMAP = "viridis"
FILLING_CMAP = "viridis"


## Discover saved simulations

The loader searches recursively below `SWEEP_DIR` for folders containing both `metadata.json` and `data.npz`. It sorts the sequence by magnetic field `B`, not by folder name, so it is robust to timestamped output folders.


In [ ]:
def find_simulation_dirs(sweep_dir: Path):
    sweep_dir = Path(sweep_dir)
    if not sweep_dir.exists():
        raise FileNotFoundError(
            f"Cannot find {sweep_dir!s}. Run this notebook from the directory containing sweep_fold, "
            "or set SWEEP_DIR to the correct path."
        )
    run_dirs = []
    for data_file in sweep_dir.rglob("data.npz"):
        run_dir = data_file.parent
        if (run_dir / "metadata.json").exists():
            run_dirs.append(run_dir)
    if not run_dirs:
        raise FileNotFoundError(f"No simulation folders with data.npz and metadata.json found below {sweep_dir!s}.")
    return sorted(run_dirs)


def read_B_from_metadata(run_dir: Path):
    with open(run_dir / "metadata.json", "r") as f:
        metadata = json.load(f)
    try:
        return float(metadata["params"]["B"]), metadata
    except KeyError as exc:
        raise KeyError(f"Could not find params['B'] in {run_dir / 'metadata.json'}") from exc


run_dirs = find_simulation_dirs(SWEEP_DIR)
print(f"Found {len(run_dirs)} saved simulations.")
print("First folder:", run_dirs[0])
print("Last folder: ", run_dirs[-1])

## Load `B`, the effective potential, the filling factor, and the chemical potential

This cell avoids loading unnecessary large arrays into memory. From each `data.npz` file it reads only `POTENTIAL_KEY`, `FILLING_KEY`, and a chemical-potential entry when available. From `metadata.json` it reads `B` and also falls back to metadata entries for `mu` if needed. Coordinates `x` and `y` are read once from the first dataset, assuming all runs use the same grid.


In [ ]:
def _maybe_scalar_float(value):
    """Convert a scalar-like value to float; return None if impossible."""
    try:
        arr = np.asarray(value)
        if arr.size == 1:
            return float(arr.reshape(-1)[0])
    except Exception:
        pass
    return None


def _recursive_find_first_scalar(d, candidate_keys):
    """Search nested dictionaries/lists/tuples for the first scalar value under candidate_keys."""
    if isinstance(d, dict):
        for key in candidate_keys:
            if key in d:
                out = _maybe_scalar_float(d[key])
                if out is not None:
                    return out
        for value in d.values():
            out = _recursive_find_first_scalar(value, candidate_keys)
            if out is not None:
                return out
    elif isinstance(d, (list, tuple)):
        for value in d:
            out = _recursive_find_first_scalar(value, candidate_keys)
            if out is not None:
                return out
    return None


def extract_mu_from_data_or_metadata(data, metadata, run_dir, strict=False):
    """Return chemical potential from data.npz or metadata.json.

    Robust behavior:
    - tries several common keys in data.npz;
    - tries the same keys recursively inside metadata.json;
    - if nothing is found, returns np.nan instead of stopping the notebook.

    Set strict=True if you want the notebook to raise an error when mu is missing.
    """
    data_mu_keys = [
        "mu", "mu_meV", "mu_eV",
        "chemical_potential", "chemical_potential_meV", "chemical_potential_eV",
        "EF", "E_F", "fermi_energy", "fermi_energy_meV",
        "U_F", "U_F_meV",
    ]

    for key in data_mu_keys:
        if key in data.files:
            out = _maybe_scalar_float(data[key])
            if out is not None:
                return out

    out = _recursive_find_first_scalar(metadata, data_mu_keys)
    if out is not None:
        return out

    msg = (
        f"Could not find chemical potential in {run_dir / 'data.npz'} or {run_dir / 'metadata.json'}. "
        "The frame will be loaded with mu = np.nan. "
        "Tried keys: " + ", ".join(data_mu_keys)
    )
    if strict:
        raise KeyError(msg)
    print("Warning:", msg)
    return np.nan


def load_effective_potential_filling_sequence(
    run_dirs,
    potential_key="U",
    filling_key="nu",
    ground_each_frame=True,
    dtype=np.float32,
):
    records = []
    x_ref = None
    y_ref = None
    shape_ref = None

    for run_dir in run_dirs:
        B, metadata = read_B_from_metadata(run_dir)
        with np.load(run_dir / "data.npz", allow_pickle=False) as data:
            if potential_key not in data.files:
                raise KeyError(
                    f"Key {potential_key!r} not found in {run_dir / 'data.npz'}. "
                    f"Available keys: {data.files}"
                )
            if filling_key not in data.files:
                raise KeyError(
                    f"Key {filling_key!r} not found in {run_dir / 'data.npz'}. "
                    f"Available keys: {data.files}"
                )

            U = np.asarray(data[potential_key], dtype=dtype)
            if ground_each_frame:
                U = U - np.nanmin(U)
            U = U.astype(dtype, copy=False)

            nu = np.asarray(data[filling_key], dtype=dtype).astype(dtype, copy=False)
            mu = extract_mu_from_data_or_metadata(data, metadata, run_dir)

            if x_ref is None:
                x_ref = np.asarray(data["x"], dtype=dtype) if "x" in data.files else np.arange(U.shape[1], dtype=dtype)
                y_ref = np.asarray(data["y"], dtype=dtype) if "y" in data.files else np.arange(U.shape[0], dtype=dtype)
                shape_ref = U.shape
            else:
                if U.shape != shape_ref:
                    raise ValueError(f"Potential grid shape mismatch in {run_dir}: {U.shape} != {shape_ref}")
                if nu.shape != shape_ref:
                    raise ValueError(f"Filling grid shape mismatch in {run_dir}: {nu.shape} != {shape_ref}")

        records.append((B, run_dir, U, nu, mu))

    records.sort(key=lambda item: item[0])
    B_values = np.array([r[0] for r in records], dtype=dtype)
    U_stack = np.stack([r[2] for r in records], axis=0).astype(dtype, copy=False)
    nu_stack = np.stack([r[3] for r in records], axis=0).astype(dtype, copy=False)
    mu_values = np.array([r[4] for r in records], dtype=dtype)
    sorted_dirs = [r[1] for r in records]
    return B_values, U_stack, nu_stack, mu_values, x_ref, y_ref, sorted_dirs


B_values, U_stack, nu_stack, mu_values, x, y, sorted_run_dirs = load_effective_potential_filling_sequence(
    run_dirs,
    potential_key=POTENTIAL_KEY,
    filling_key=FILLING_KEY,
    ground_each_frame=GROUND_EACH_FRAME,
    dtype=DTYPE,
)

print("B_values.shape:", B_values.shape)
print("U_stack.shape: ", U_stack.shape, "  # (nB, Ny, Nx)")
print("nu_stack.shape:", nu_stack.shape, " # (nB, Ny, Nx)")
print("mu_values.shape:", mu_values.shape)
print("x.shape, y.shape:", x.shape, y.shape)
print(f"Approx. memory used by U_stack:  {U_stack.nbytes/1024**2:.2f} MB")
print(f"Approx. memory used by nu_stack: {nu_stack.nbytes/1024**2:.2f} MB")
print(f"B range:  {B_values.min():.6g} -> {B_values.max():.6g} T")
if np.isfinite(mu_values).any():
    print(f"mu range: {np.nanmin(mu_values):.6g} -> {np.nanmax(mu_values):.6g}")
else:
    print("mu_values are all NaN: no chemical-potential key was found in the sweep frames.")


## Optionally cache the lightweight sequence

The cache contains only `B_values`, `U_stack`, `nu_stack`, `mu_values`, `x`, and `y`. This is much smaller than the full simulation archive because it excludes densities, kernels, compressibility, histories, etc.


In [ ]:
np.savez_compressed(
    CACHE_FILE,
    B_values=B_values,
    U_stack=U_stack,
    nu_stack=nu_stack,
    mu_values=mu_values,
    x=x,
    y=y,
    potential_key=np.array(POTENTIAL_KEY),
    filling_key=np.array(FILLING_KEY),
    ground_each_frame=np.array(GROUND_EACH_FRAME),
)
print(f"Saved lightweight cache to {CACHE_FILE.resolve()}")


## Inspect representative frames

In [ ]:
def plot_potential_frame(index, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(7.0, 3.8))
    extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]
    im = ax.imshow(
        U_stack[index],
        origin="lower",
        extent=extent,
        aspect="auto",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"{POTENTIAL_KEY}, B = {B_values[index]:.4g} T")
    ax.set_xlabel("x [nm]")
    ax.set_ylabel("y [nm]")
    cb = fig.colorbar(im, ax=ax)
    label = f"{POTENTIAL_KEY} - min({POTENTIAL_KEY}) [meV]" if GROUND_EACH_FRAME else f"{POTENTIAL_KEY} [meV]"
    cb.set_label(label)
    fig.tight_layout()
    return fig, ax

# Use common color limits for meaningful visual comparison across B.
#vmin = float(np.nanpercentile(U_stack, 1))
#vmax = float(np.nanpercentile(U_stack, 99))
vmin = 0
vmax = 5

for idx in np.linspace(0, len(B_values)-1, min(4, len(B_values)), dtype=int):
    plot_potential_frame(idx, vmin=vmin, vmax=vmax)
plt.show()

## Inspect representative filling-factor frames


In [ ]:
def plot_filling_frame(index, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(7.0, 3.8))
    extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]
    im = ax.imshow(
        nu_stack[index],
        origin="lower",
        extent=extent,
        aspect="auto",
        cmap=FILLING_CMAP,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"{FILLING_KEY}, B = {B_values[index]:.4g} T")
    ax.set_xlabel("x [nm]")
    ax.set_ylabel("y [nm]")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(r"filling factor $\nu(x,y)$")
    fig.tight_layout()
    return fig, ax

# Use common color limits for meaningful visual comparison across B.
nu_vmin = float(np.nanpercentile(nu_stack, 1))
nu_vmax = float(np.nanpercentile(nu_stack, 99))
# Alternatively, use fixed limits, for example:
# nu_vmin = 0
# nu_vmax = 8

for idx in np.linspace(0, len(B_values)-1, min(4, len(B_values)), dtype=int):
    plot_filling_frame(idx, vmin=nu_vmin, vmax=nu_vmax)
plt.show()


## Chemical potential during the B sweep


In [ ]:
# Optional: compare mu(B) with approximate Kwant bandwidth
m_eff = 0.067   # GaAs
a_nm = abs(x[1]-x[0])     # set this to the Kwant lattice spacing used in transport

HBAR2_OVER_2ME_MEV_NM2 = 38.0998212  # hbar^2/(2 m_e) in meV nm^2

t_meV = HBAR2_OVER_2ME_MEV_NM2 / (m_eff * a_nm**2)
bandwidth_meV = 8 * t_meV
print("bandwidth leads = %.1f meV" % bandwidth_meV)

finite_mu = np.isfinite(mu_values)

fig, ax = plt.subplots(figsize=(6.5, 4.0))
if finite_mu.any():
    ax.plot(B_values[finite_mu], mu_values[finite_mu], marker="o")
    ax.set_ylabel(r"chemical potential $\mu$ [simulation units]")
    ax.set_title(r"Chemical potential during the $B$ sweep")
else:
    ax.plot(B_values, np.zeros_like(B_values), alpha=0.0)
    ax.text(
        0.5, 0.5,
        "No chemical-potential values found\n(mu_values are NaN)",
        ha="center", va="center", transform=ax.transAxes,
    )
    ax.set_ylabel(r"chemical potential $\mu$")
    ax.set_title(r"Chemical potential during the $B$ sweep")
#ax.plot(B_values, np.ones(len(B_values))*bandwidth_meV, linestyle='--' )
ax.set_xlabel("B [T]")
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig.savefig(MU_PLOT_PNG, dpi=DPI)
print(f"Saved chemical-potential plot to {MU_PLOT_PNG.resolve()}")
plt.show()


In [ ]:
#plt.plot(B_values, np.gradient(mu_values), marker="o")
#plt.ylim(0,.15)


In [ ]:
index = 0

n_phi = 1./(2*np.pi*25**2./B_values[index])
n_e_res = nu_stack[index]*n_phi * 1e3

iy = len(y) // 2
ix = len(x) // 2

nxx = np.argmin(np.abs(x-750))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x, n_e_res[iy, :], label="y=0 cut")
ax.plot(y, n_e_res[:, ix], label="x=0 cut")

print(n_e_res[iy, ix])

ax.set_xlabel("coordinate [nm]")
ax.set_ylabel(r"$n_e$")
ax.grid(True)
ax.legend()

fig.tight_layout()
print( n_e_res[iy, ix]- n_e_res[iy, nxx])

In [ ]:
index = 0

U_res = U_stack[index]/(1.76*B_values[index])

iy = len(y) // 2
ix = len(x) // 2

nxx = np.argmin(np.abs(x-750))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x, U_res[iy, :], label="y=0 cut", marker='o')
ax.plot(y, U_res[:, ix], label="x=0 cut", marker='o')
ax.axhline(mu_values[index], color='green')


ax.set_xlabel("coordinate [nm]")
ax.set_ylabel(r"$U/(\hbar \omega_B)$")
ax.grid(True)
ax.legend()
ax.set_ylim(0,12)

fig.tight_layout()


In [ ]:
print(mu_values[index]-U_res[:, ix])

In [ ]:
index = 10

EB = 1.76*B_values[index]
U_res = (U_stack[index]-np.amin(U_stack[index]))/(EB)

iy = len(y) // 2
ix = len(x) // 2

nxx = np.argmin(np.abs(x-750))
mu_frame = mu_values[index]-np.amin(U_stack[index])
a_eff = a_nm

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(y, U_res[:, ix]*EB, label="x=0 cut", marker='o')
for nee in range(12,14):
    a_eff = a_nm
    lB = 25.658206/np.sqrt(B_values[index])
    ll_effective = nee
    JJ = 38.0998212/0.067/a_eff**2
    #dE = - EB**2/(32*JJ) * ( 2*ll_effective**2 + 2*ll_effective + 1 )*1
    dE = - EB/16. * (a_nm/lB)**2. * (2*ll_effective**2 + 2*ll_effective + 1)
    #print(dE)
    print( (U_res[iy, ix]+.5+nee + dE/EB - (mu_values[index]-np.amin(U_stack[index]))/EB)*EB, (EB*(.5+nee)+dE)/EB )
    ax.plot(y, (U_res[:, ix]+.5+nee + dE/EB)*EB )

ax.axhline( (mu_values[index]-np.amin(U_stack[index]))/EB*EB , color='green')


ax.set_xlabel("coordinate [nm]")
ax.set_ylabel(r"$U/(\hbar \omega_B)$")
ax.grid(True)
ax.legend()
ax.set_ylim(10,13.)

fig.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 3.2))

iy = len(y) // 2
ix = len(x) // 2

# Prepare the lines once
line_U, = ax.plot([], [], label="x=0 cut", marker='o', markersize=3)

LL_lines = []
for nee in range(16):
    line, = ax.plot([], [], lw=1)
    LL_lines.append(line)

mu_line = ax.axhline(0, color='green', lw=2, label=r"$\mu$")

ax.set_xlabel("coordinate [nm]")
ax.set_ylabel(r"$U/(\hbar \omega_B)$")
ax.set_xlim(y.min(), y.max())
ax.set_ylim(0, 16)
ax.grid(True)
ax.legend()

title = ax.set_title("")


def update(index):

    EB = 1.76 * B_values[index]

    U_res = (
        U_stack[index] - np.amin(U_stack[index])
    ) / EB

    mu_res = (
        mu_values[index] - np.amin(U_stack[index])
    ) / EB

    # Confining potential
    line_U.set_data(y, U_res[:, ix])

    # Landau levels: U + (n + 1/2) hbar omega_c
    for nee, line in enumerate(LL_lines):
        a_eff = a_nm
        ll_effective = nee
        JJ = 38.0998212/0.067/a_eff**2
        dE = - EB**2/(32*JJ) * ( 2*ll_effective**2 + 2*ll_effective + 1 )
        line.set_data(
            y,
            U_res[:, ix] + 0.5 + nee + dE/EB
        )

    # Chemical potential
    ll_effective = np.around(mu_res-.5, decimals=0)         # correct \mu by finite discretization error
    JJ = 38.0998212/0.067/a_eff**2
    dE = - EB**2/(32*JJ) * ( 2*ll_effective**2 + 2*ll_effective + 1 )*0
    dE0 = - EB**2/(32*JJ) * ( 2*(ll_effective-1)**2 + 2*(ll_effective-1) + 1 )*0
    mu_res1 = mu_res + (dE+dE0)/EB/2*0
    mu_line.set_ydata([mu_res1, mu_res1])

    title.set_text(
        f"index = {index} / {len(B_values)-1}   "
        f"B = {B_values[index]:.3f} T"
    )

    return [line_U, mu_line, title] + LL_lines


anim = FuncAnimation(
    fig,
    update,
    frames=len(B_values),
    interval=120,   # milliseconds per frame
    blit=False,
    repeat=True
)

plt.close(fig)

HTML(anim.to_jshtml())

## Build the filling-factor animation


In [ ]:
fig_nu, ax_nu = plt.subplots(figsize=(7.0, 3.8))
extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]

im_nu = ax_nu.imshow(
    nu_stack[0],
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap=FILLING_CMAP,
    vmin=nu_vmin,
    vmax=nu_vmax,
)

ax_nu.set_xlabel("x [nm]")
ax_nu.set_ylabel("y [nm]")
title_nu = ax_nu.set_title("")
cb_nu = fig_nu.colorbar(im_nu, ax=ax_nu)
cb_nu.set_label(r"filling factor $\nu(x,y)$")
fig_nu.tight_layout()


def update_nu(frame):
    im_nu.set_data(nu_stack[frame])
    title_nu.set_text(f"{FILLING_KEY}, B = {B_values[frame]:.4g} T   frame {frame+1}/{len(B_values)}")
    return im_nu, title_nu

anim_nu = FuncAnimation(fig_nu, update_nu, frames=len(B_values), interval=1000/FPS, blit=False)
plt.close(fig_nu)

display(HTML(anim_nu.to_jshtml()))


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

fig_nu, ax_nu = plt.subplots(figsize=(7.0, 3.8))
extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]

im_nu = ax_nu.imshow(
    nu_stack[0],
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap=FILLING_CMAP,
    vmin=nu_vmin,
    vmax=nu_vmax,
)

ax_nu.set_xlabel("x [nm]")
ax_nu.set_ylabel("y [nm]")
title_nu = ax_nu.set_title("")

cb_nu = fig_nu.colorbar(im_nu, ax=ax_nu)
cb_nu.set_label(r"filling factor $\nu(x,y)$")

# Integer filling-factor contour levels.
nu_integer_levels = np.arange(
    np.ceil(nu_vmin),
    np.floor(nu_vmax) + 1,
    1.0,
)

# Initial contours.
contour_nu = ax_nu.contour(
    x,
    y,
    nu_stack[0],
    levels=nu_integer_levels,
    linewidths=2.0,
    color='black',
)

fig_nu.tight_layout()


def update_nu(frame):
    global contour_nu

    im_nu.set_data(nu_stack[frame])

    # Remove previous contour set.
    contour_nu.remove()

    # Draw new integer-filling contours.
    contour_nu = ax_nu.contour(
        x,
        y,
        nu_stack[frame],
        levels=nu_integer_levels,
        linewidths=2.0,
        color='black',
    )

    title_nu.set_text(
        f"{FILLING_KEY}, B = {B_values[frame]:.4g} T   "
        f"frame {frame+1}/{len(B_values)}"
    )

    return im_nu, title_nu


anim_nu = FuncAnimation(
    fig_nu,
    update_nu,
    frames=len(B_values),
    interval=1000 / FPS,
    blit=False,
)

plt.close(fig_nu)

display(HTML(anim_nu.to_jshtml()))

## Export filling-factor video

This cell first tries to save an MP4 with `ffmpeg`. If `ffmpeg` is not installed, it falls back to a GIF.


In [ ]:
try:
    writer = FFMpegWriter(fps=FPS, bitrate=2400)
    anim_nu.save(FILLING_VIDEO_MP4, writer=writer, dpi=DPI)
    print(f"Saved filling-factor MP4 video to {FILLING_VIDEO_MP4.resolve()}")
except Exception as exc:
    print("Filling-factor MP4 export failed, probably because ffmpeg is unavailable.")
    print("Error:", repr(exc))
    writer = PillowWriter(fps=FPS)
    anim_nu.save(FILLING_VIDEO_GIF, writer=writer, dpi=DPI)
    print(f"Saved filling-factor GIF animation to {FILLING_VIDEO_GIF.resolve()}")


## Build the animation

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 6.0))
extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]

im = ax.imshow(
    U_stack[0],
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=3,
)

ax.set_xlabel("x [nm]")
ax.set_ylabel("y [nm]")
title = ax.set_title("")
cb = fig.colorbar(im, ax=ax)
cb_label = f"{POTENTIAL_KEY} - min({POTENTIAL_KEY}) [meV]" if GROUND_EACH_FRAME else f"{POTENTIAL_KEY} [meV]"
cb.set_label(cb_label)
fig.tight_layout()


def update(frame):
    im.set_data(U_stack[frame])
    title.set_text(f"{POTENTIAL_KEY}, B = {B_values[frame]:.4g} T   frame {frame+1}/{len(B_values)}")
    return im, title

anim = FuncAnimation(fig, update, frames=len(B_values), interval=1000/FPS, blit=False)
plt.close(fig)

display(HTML(anim.to_jshtml()))

In [ ]:
nxr = np.argmin(np.abs(x-500))
nxl = np.argmin(np.abs(x))
ny0 = np.argmin(np.abs(y))
print(nxr, nxl, ny0)
dU0 = U_stack[:,ny0,nxl]-U_stack[:,ny0,nxr]
plt.plot( B_values , abs(dU0/(1.75*B_values)), marker='o')
#plt.ylim(0,2)
plt.grid()

In [ ]:
nB = np.argmin(np.abs(B_values-1.1))
ny0 = np.argmin(np.abs(y))
nxl = np.argmin(np.abs(x-700))
nxcenter = np.argmin(np.abs(x))

cutU = U_stack[nB, ny0, :] - np.amin(U_stack[nB, :, :])
mu_B = mu_values[nB] - np.amin(U_stack[nB, :, :])
filling_B_left = nu_stack[nB, ny0, nxl]
filling_B_center = nu_stack[nB, ny0, nxcenter]


delta_mu = mu_B - (1.76*B_values[nB]*.5 + (filling_B_left-1)*1.76*B_values[nB])

print(filling_B_left-filling_B_center, delta_mu/(1.76*B_values[nB]))

plt.plot(x, cutU/(1.76*B_values[nB]), label='U(x) at y=0')
plt.axhline(delta_mu/(1.76*B_values[nB]), color='r', linestyle='--', label=r'$\delta \mu(B)$')
plt.xlabel('x [nm]')
plt.ylabel(f'{POTENTIAL_KEY} in cyclotron')
plt.title(f'B = {B_values[nB]:.4g} T')
plt.legend()
plt.ylim(0,2)
plt.grid()
plt.tight_layout()

In [ ]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
VIDEO_FILE = "output_sweep/potential_cut_vs_B.mp4"
VIDEO_FPS = 8
VIDEO_DPI = 150
ny0 = np.argmin(np.abs(y))
nxl = np.argmin(np.abs(x - 700))
nxcenter = np.argmin(np.abs(x))
# ------------------------------------------------------------
# Precompute quantities for the full B sweep
# ------------------------------------------------------------
cutU_all = []
delta_mu_all = []
delta_filling_all = []
for nB in range(len(B_values)):
    B = B_values[nB]
    Umin = np.amin(U_stack[nB, :, :])
    cutU = U_stack[nB, ny0, :] - Umin
    mu_B = mu_values[nB] - Umin
    filling_B_left = nu_stack[nB, ny0, nxl]
    filling_B_center = nu_stack[nB, ny0, nxcenter]
    
    delta_mu = mu_B - (1.76 * B * 0.5 + (filling_B_left - 1) * 1.76 * B)
    
    cutU_all.append(cutU / (1.76 * B))
    delta_mu_all.append(delta_mu / (1.76 * B))
    delta_filling_all.append(filling_B_left - filling_B_center)
cutU_all = np.asarray(cutU_all)
delta_mu_all = np.asarray(delta_mu_all)
delta_filling_all = np.asarray(delta_filling_all)
# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
line_U, = ax.plot(
    x,
    cutU_all[0],
    lw=2,
    label=r"$U(x,y=0)/\hbar\omega_c$"
)
line_mu = ax.axhline(
    delta_mu_all[0],
    linestyle="--",
    lw=2,
    label=r"$\delta\mu(B)/\hbar\omega_c$"
)
ax.set_xlabel("x [nm]")
ax.set_ylabel(f"{POTENTIAL_KEY} in cyclotron units")
ax.set_ylim(0, 8)
ax.set_xlim(np.min(x), np.max(x))
ax.grid()
ax.legend(loc="upper right")
title = ax.set_title("")
# Optional text box showing additional quantities
info_text = ax.text(
    0.02,
    0.96,
    "",
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox=dict(boxstyle="round", alpha=0.8)
)
# ------------------------------------------------------------
# Animation update function
# ------------------------------------------------------------
def update(nB):
    B = B_values[nB]
    line_U.set_ydata(cutU_all[nB])
    # axhline is internally a 2-point horizontal line
    line_mu.set_ydata([delta_mu_all[nB], delta_mu_all[nB]])
    title.set_text(
        f"B = {B:.4g} T   "
        f"({nB + 1}/{len(B_values)})"
    )
    info_text.set_text(
        rf"$\Delta\nu = \nu(x=700)-\nu(x=0)"
        f"\n= {delta_filling_all[nB]:.4f}"
        "\n"
        rf"$\delta\mu/\hbar\omega_c$"
        f" = {delta_mu_all[nB]:.4f}"
    )
    return line_U, line_mu, title, info_text
# ------------------------------------------------------------
# Create animation
# ------------------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(B_values),
    interval=1000 / VIDEO_FPS,
    blit=False
)
plt.close(fig)
# ------------------------------------------------------------
# Save MP4
# ------------------------------------------------------------
anim.save(
    VIDEO_FILE,
    writer="ffmpeg",
    fps=VIDEO_FPS,
    dpi=VIDEO_DPI
)
# ------------------------------------------------------------
# Display with play/pause/seek controls in Jupyter
# ------------------------------------------------------------
display(
    Video(
        VIDEO_FILE,
        embed=True,
        html_attributes="controls loop"
    )
)

## Export video

This cell first tries to save an MP4 with `ffmpeg`. If `ffmpeg` is not installed, it falls back to a GIF.


In [ ]:
try:
    writer = FFMpegWriter(fps=FPS, bitrate=2400)
    anim.save(VIDEO_MP4, writer=writer, dpi=DPI)
    print(f"Saved MP4 video to {VIDEO_MP4.resolve()}")
except Exception as exc:
    print("MP4 export failed, probably because ffmpeg is unavailable.")
    print("Error:", repr(exc))
    writer = PillowWriter(fps=FPS)
    anim.save(VIDEO_GIF, writer=writer, dpi=DPI)
    print(f"Saved GIF animation to {VIDEO_GIF.resolve()}")

## Notes

- The original sweep notebook saves many arrays per run: density `n`, filling `nu`, potentials, compressibility, donor profile, masks, Coulomb kernels, and convergence history.
- This notebook deliberately loads only `U`, `nu`, `mu`, and `B` for the full sweep, plus `x` and `y` once for plotting.
- The chemical-potential key is detected flexibly: the loader first checks `data.npz` and then `metadata.json` for common names such as `mu`, `mu_meV`, and `chemical_potential`.
- If the sweep folders are very large, you can delete the cache file and regenerate it from `sweep_fold/` at any time.


In [ ]:
s = 20
width = 200
d = np.linspace(0,400)
f = 0.5*(np.tanh( (d-100)/s) - np.tanh(((d-100)-width)/s))
plt.plot(d,f)